# SMS Smishing Detection — ML Model

This notebook trains and evaluates multiple classifiers to detect SMS phishing (smishing) messages.

**Pipeline:**
1. Load dataset
2. Feature engineering (text features)
3. NLP preprocessing with Sentence-BERT (imported from `NLP_smish.py`)
4. Train 5 classifiers with GridSearchCV + StratifiedKFold
5. Build weighted soft-voting ensemble
6. Compare all 6 candidates and save the best model

**Feature vector per message:** 384 (SBERT) + 3 (url/email/phone flags) + 7 (text features) = **394 dimensions**

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)

# Import shared utilities from the NLP preprocessing module
from NLP_smish import (
    preprocess_text, extract_features_from_text, encode_flags,
    SBERT_MODEL_NAME, BATCH_SIZE
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

# ── Configuration ────────────────────────────────────────────────────────────
DATASET_PATH  = 'dataset/dataset_with_new_smish_without_spam.csv'
RANDOM_STATE  = 42
TEST_SIZE     = 0.2
CV_FOLDS      = 5
MODEL_PATH    = 'smishing_detector.pkl'
ENCODER_PATH  = 'label_encoder.pkl'

# Urgency keywords that are common in phishing messages
URGENCY_WORDS = [
    'free', 'win', 'prize', 'claim', 'click', 'verify', 'urgent',
    'account', 'bank', 'password', 'confirm', 'expire', 'limited',
    'offer', 'reward', 'cash', 'selected', 'congratulations',
    'act now', 'immediately'
]

PALETTE = ['#2196F3', '#F44336']   # blue = ham, red = smish

print('All imports successful.')
print(f'SBERT model: {SBERT_MODEL_NAME}')

## 2. Load Dataset & Exploratory Data Analysis

In [ ]:
df = pd.read_csv(DATASET_PATH)

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nLabel distribution (%):')
print((df['label'].value_counts(normalize=True) * 100).round(2))

df.head()

## 2. Load Dataset & Exploratory Data Analysis

The following six graphs explore the dataset structure before training — class balance, message length patterns, flag co-occurrence, most frequent words, and feature correlations.

In [ ]:
# ── EDA constants ─────────────────────────────────────────────────────────────
os.makedirs('graphs', exist_ok=True)

GRAPH_PALETTE = {'ham': '#2196F3', 'smish': '#F44336'}

STOPWORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you',
    'your','yours','yourself','he','him','his','she','her','hers',
    'it','its','they','them','their','what','which','who','this',
    'that','these','those','am','is','are','was','were','be',
    'been','being','have','has','had','do','does','did','will',
    'would','should','can','could','may','might','shall','must',
    'a','an','the','and','but','if','or','as','of','at','by',
    'for','with','about','to','from','in','on','up','so','no',
    'not','u','r','ur','2','4','get','just','now','got','go',
    'know','like','one','come','good','want','im','dont','ill',
    'ok','yeah','hi','hey','lol','ya','da','na','oh','gt','lt'
}

# Prepare EDA dataframe with normalized flags and all derived features
df_eda = df.copy()
for col in ['url', 'email', 'phone']:
    df_eda[col] = df_eda[col].astype(str).str.lower().map({'yes': 1, 'no': 0}).fillna(0).astype(int)

df_eda['label_binary']       = (df_eda['label'] == 'smish').astype(int)
df_eda['msg_length']         = df_eda['message'].str.len()
df_eda['word_count']         = df_eda['message'].str.split().str.len()
df_eda['uppercase_ratio']    = df_eda['message'].apply(
    lambda m: sum(c.isupper() for c in str(m)) / max(len(str(m)), 1))
df_eda['digit_count']        = df_eda['message'].apply(lambda m: sum(c.isdigit() for c in str(m)))
df_eda['special_char_count'] = df_eda['message'].apply(lambda m: sum(c in '!$#*@' for c in str(m)))
df_eda['urgency_score']      = df_eda['message'].str.lower().apply(
    lambda m: sum(w in m.split() for w in URGENCY_WORDS))
df_eda['exclamation_count']  = df_eda['message'].str.count('!')

print(f'EDA dataframe prepared: {df_eda.shape[0]:,} rows')
print(f'Label distribution:\n{df_eda["label"].value_counts().to_string()}')

In [ ]:
def top_words(text_series, n=20):
    """Count word frequencies, excluding stopwords and short tokens."""
    words = []
    for msg in text_series.dropna():
        for w in str(msg).lower().split():
            w = ''.join(c for c in w if c.isalpha())
            if w and w not in STOPWORDS and len(w) > 2:
                words.append(w)
    return pd.Series(words).value_counts().head(n)

ham_words   = top_words(df_eda[df_eda['label'] == 'ham']['message'])
smish_words = top_words(df_eda[df_eda['label'] == 'smish']['message'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.barh(ham_words.index[::-1], ham_words.values[::-1],
         color=GRAPH_PALETTE['ham'], edgecolor='white', alpha=0.9)
ax1.set_title('Top 20 Words — Ham Messages', fontsize=13, fontweight='bold')
ax1.set_xlabel('Frequency', fontsize=11)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.barh(smish_words.index[::-1], smish_words.values[::-1],
         color=GRAPH_PALETTE['smish'], edgecolor='white', alpha=0.9)
ax2.set_title('Top 20 Words — Smish Messages', fontsize=13, fontweight='bold')
ax2.set_xlabel('Frequency', fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig.suptitle('Most Frequent Words by Class (stopwords removed)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graphs/04_top_words_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

### Graph 5 — Message Length Boxplot

Box-and-whisker view showing the median, IQR, and outliers for each class. Useful for spotting that smish messages tend to be longer with more extreme outliers.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ham_len   = df_eda[df_eda['label'] == 'ham']['msg_length']
smish_len = df_eda[df_eda['label'] == 'smish']['msg_length']

bp = ax.boxplot([ham_len, smish_len],
                patch_artist=True,
                tick_labels=['Ham', 'Smish'],
                medianprops=dict(color='white', linewidth=2.5),
                whiskerprops=dict(linewidth=1.5),
                capprops=dict(linewidth=1.5),
                flierprops=dict(marker='o', markersize=3, alpha=0.25))

bp['boxes'][0].set_facecolor(GRAPH_PALETTE['ham'])
bp['boxes'][1].set_facecolor(GRAPH_PALETTE['smish'])

for i, lengths in enumerate([ham_len, smish_len], 1):
    med = lengths.median()
    ax.text(i, med + 8, f'Median: {int(med)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Message Length Distribution by Class (with Outliers)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Class', fontsize=12)
ax.set_ylabel('Message Length (characters)', fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('graphs/05_message_length_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

### Graph 6 — Feature Correlation Heatmap

Shows Pearson correlations between all engineered features and the binary label. High positive correlation with `label_binary` means the feature is a strong smish signal.

In [ ]:
eda_feature_cols = [
    'msg_length', 'word_count', 'uppercase_ratio',
    'digit_count', 'special_char_count', 'urgency_score',
    'exclamation_count', 'url', 'email', 'phone', 'label_binary'
]

corr = df_eda[eda_feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, linecolor='white',
            square=True, cbar_kws={'shrink': 0.8})

ax.set_title('Feature Correlation Heatmap\n(label_binary: 0 = ham, 1 = smish)',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('graphs/06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering

We add 7 hand-crafted text features that capture phishing signal beyond what SBERT embeddings encode:

| Feature | Rationale |
|---|---|
| `msg_length` | Phishing messages tend to be longer (calls-to-action) |
| `word_count` | Related to message length |
| `uppercase_ratio` | SHOUTING is a phishing pattern |
| `digit_count` | Phone numbers, short codes, amounts |
| `special_char_count` | `!`, `$`, `#`, `*`, `@` — urgency/money signals |
| `urgency_score` | Count of phishing urgency keywords |
| `exclamation_count` | Urgency indicator |

In [ ]:
def engineer_features(df_input):
    """
    Add 7 text-derived features to the DataFrame.
    Works on both training and new single-message DataFrames.
    """
    d = df_input.copy()
    d['msg_length']         = d['message'].str.len()
    d['word_count']         = d['message'].str.split().str.len()
    d['uppercase_ratio']    = d['message'].apply(
        lambda m: sum(c.isupper() for c in str(m)) / max(len(str(m)), 1))
    d['digit_count']        = d['message'].apply(lambda m: sum(c.isdigit() for c in str(m)))
    d['special_char_count'] = d['message'].apply(lambda m: sum(c in '!$#*@' for c in str(m)))
    d['urgency_score']      = d['message'].str.lower().apply(
        lambda m: sum(w in m.split() for w in URGENCY_WORDS))
    d['exclamation_count']  = d['message'].str.count('!')
    return d

df = engineer_features(df)

# Normalize flag columns: yes/Yes/no/No → 1/0 (using encode_flags from NLP_smish)
for col in ['url', 'email', 'phone']:
    df[col] = df[col].astype(str).str.lower().map({'yes': 1, 'no': 0}).fillna(0).astype(int)

print('Feature engineering complete.')
print(f'\nNew columns added:')
text_feature_cols = ['msg_length', 'word_count', 'uppercase_ratio',
                     'digit_count', 'special_char_count', 'urgency_score', 'exclamation_count']
display(df[['label'] + text_feature_cols].groupby('label').mean().round(3))

## 4. Train / Test Split & Label Encoding

In [ ]:
# Encode labels: ham → 0, smish → 1
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])
print(f'Label mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}')

# Keep only the columns we need for the split
feature_df = df[['message', 'url', 'email', 'phone'] + text_feature_cols]

# 80/20 stratified split — ensures same class ratio in both sets
X_train_df, X_test_df, y_train, y_test = train_test_split(
    feature_df, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'\nTrain set size: {len(X_train_df):,} ({len(X_train_df)/len(df)*100:.1f}%)')
print(f'Test set size:  {len(X_test_df):,} ({len(X_test_df)/len(df)*100:.1f}%)')
print(f'\nTrain label distribution:')
vals, cnts = np.unique(y_train, return_counts=True)
for v, c in zip(vals, cnts):
    print(f'  {label_encoder.inverse_transform([v])[0]}: {c:,} ({c/len(y_train)*100:.1f}%)')

## 5. NLP Preprocessing with Sentence-BERT

Sentence-BERT encodes each SMS message into a 384-dimensional semantic embedding vector. These embeddings capture the meaning of the text — phishing messages have distinctive patterns (urgency, impersonation, calls-to-action) that SBERT can distinguish.

In [ ]:
# Initialize Sentence-BERT (downloads model on first run, then cached)
print(f'Loading Sentence-BERT model: {SBERT_MODEL_NAME}...')
sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
print('Model loaded.')

# Preprocess text (lowercase + whitespace normalization)
X_train_text = X_train_df['message'].apply(preprocess_text).tolist()
X_test_text  = X_test_df['message'].apply(preprocess_text).tolist()

print(f'\nGenerating SBERT embeddings for {len(X_train_text):,} training messages...')
X_train_sbert = sbert_model.encode(
    X_train_text,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'\nGenerating SBERT embeddings for {len(X_test_text):,} test messages...')
X_test_sbert = sbert_model.encode(
    X_test_text,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'\nSBERT embedding shape (train): {X_train_sbert.shape}')

In [ ]:
# Extract binary flag features (url, email, phone) from the message text
X_train_flags = X_train_df[['url', 'email', 'phone']].values.astype(float)
X_test_flags  = X_test_df[['url', 'email', 'phone']].values.astype(float)

# Extract the 7 engineered text features
X_train_text_feats = X_train_df[text_feature_cols].values.astype(float)
X_test_text_feats  = X_test_df[text_feature_cols].values.astype(float)

# Combine: SBERT (384) + flags (3) + text features (7) = 394 dimensions
X_train = np.hstack([X_train_sbert, X_train_flags, X_train_text_feats])
X_test  = np.hstack([X_test_sbert,  X_test_flags,  X_test_text_feats])

print('Feature matrix assembled.')
print(f'Train shape: {X_train.shape}  (384 SBERT + 3 flags + 7 text = {X_train.shape[1]})')
print(f'Test shape:  {X_test.shape}')

## 6. Model Evaluation Helper

A custom `model_kpi()` function that returns a metrics DataFrame and confusion matrix — following the same convention used in the Titanic examples.

In [ ]:
def model_kpi(model_name, y_true, y_pred, y_proba=None, show_plots=True):
    """
    Compute and display evaluation metrics for a classifier.

    Returns:
        metrics_df  — DataFrame with Accuracy, Precision, Recall, F1, ROC-AUC, FPR, FNR
        cm          — confusion matrix (numpy array)
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        'Model':     model_name,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred), 4),
        'Recall':    round(recall_score(y_true, y_pred), 4),
        'F1':        round(f1_score(y_true, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_true, y_proba) if y_proba is not None else float('nan'), 4),
        'FPR':       round(fp / (fp + tn), 4),   # false positive rate (ham flagged as smish)
        'FNR':       round(fn / (fn + tp), 4),   # false negative rate (smish missed)
    }
    metrics_df = pd.DataFrame([metrics])

    if show_plots:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Confusion matrix heatmap
        ax = axes[0]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=label_encoder.classes_,
                    yticklabels=label_encoder.classes_,
                    linewidths=0.5)
        ax.set_title(f'{model_name}\nConfusion Matrix', fontsize=12, fontweight='bold')
        ax.set_xlabel('Predicted', fontsize=10)
        ax.set_ylabel('Actual', fontsize=10)

        # ROC curve
        ax = axes[1]
        if y_proba is not None:
            fpr, tpr, _ = roc_curve(y_true, y_proba)
            auc = metrics['ROC-AUC']
            ax.plot(fpr, tpr, color='#0366d6', lw=2, label=f'AUC = {auc:.4f}')
            ax.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
            ax.fill_between(fpr, tpr, alpha=0.1, color='#0366d6')
            ax.set_xlabel('False Positive Rate', fontsize=10)
            ax.set_ylabel('True Positive Rate', fontsize=10)
            ax.set_title(f'{model_name}\nROC Curve', fontsize=12, fontweight='bold')
            ax.legend(fontsize=10)
        else:
            ax.text(0.5, 0.5, 'Probabilities not available',
                    ha='center', va='center', transform=ax.transAxes)

        plt.tight_layout()
        plt.show()

    return metrics_df, cm

print('model_kpi() helper defined.')

## 7. Model Definitions

Five classifiers, each wrapped in `make_pipeline(StandardScaler(), classifier)` to prevent data leakage (scaler is fit only on train data).

`class_weight='balanced'` compensates for the 62/38 ham/smish imbalance — the minority class (smish) is weighted more heavily so the model doesn't simply default to predicting "ham".

In [ ]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

models = {
    'Logistic Regression': {
        'pipeline': make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=RANDOM_STATE)
        ),
        'params': {
            'logisticregression__C': [0.01, 0.1, 1, 10]
        }
    },
    'SVM': {
        'pipeline': make_pipeline(
            StandardScaler(),
            SVC(probability=True, class_weight='balanced', random_state=RANDOM_STATE)
        ),
        'params': {
            'svc__C':      [0.1, 1, 10],
            'svc__kernel': ['rbf', 'linear']
        }
    },
    'Random Forest': {
        'pipeline': make_pipeline(
            RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE)
        ),
        'params': {
            'randomforestclassifier__n_estimators': [100, 200],
            'randomforestclassifier__max_depth':    [None, 20]
        }
    },
    'XGBoost': {
        'pipeline': make_pipeline(
            XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE,
                          verbosity=0)
        ),
        'params': {
            'xgbclassifier__n_estimators':  [100, 200],
            'xgbclassifier__max_depth':     [4, 6],
            'xgbclassifier__learning_rate': [0.05, 0.1]
        }
    },
    'MLP': {
        'pipeline': make_pipeline(
            StandardScaler(),
            MLPClassifier(max_iter=300, early_stopping=True,
                          random_state=RANDOM_STATE)
        ),
        'params': {
            'mlpclassifier__hidden_layer_sizes': [(256, 128), (512, 256)],
            'mlpclassifier__alpha':              [0.0001, 0.001]
        }
    }
}

print(f'{len(models)} classifiers defined.')
for name in models:
    n_configs = 1
    for v in models[name]['params'].values():
        n_configs *= len(v)
    print(f'  {name}: {n_configs} hyperparameter configurations × {CV_FOLDS} folds')

## 8. Train & Tune All Models

GridSearchCV finds the best hyperparameter combination for each model using 5-fold cross-validation, optimising **F1 score** (rather than accuracy) to minimise both false positives and false negatives.

In [ ]:
trained_models = {}   # stores best fitted pipeline per model name
all_metrics    = []   # collects metrics rows for the comparison table

for name, config in models.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print(f'{"="*60}')

    grid = GridSearchCV(
        estimator=config['pipeline'],
        param_grid=config['params'],
        scoring='f1',
        cv=cv,
        refit=True,
        return_train_score=True,
        n_jobs=-1,
        verbose=0
    )
    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    trained_models[name] = best

    # Evaluate on test set
    y_pred  = best.predict(X_test)
    y_proba = best.predict_proba(X_test)[:, 1]

    # Overfitting check: compare best CV train score vs CV validation score
    best_idx  = grid.best_index_
    train_f1  = grid.cv_results_['mean_train_score'][best_idx]
    val_f1    = grid.cv_results_['mean_test_score'][best_idx]
    gap       = train_f1 - val_f1
    overfit   = 'OVERFITTING' if gap > 0.05 else 'OK'

    print(f'Best params:     {grid.best_params_}')
    print(f'CV train F1:     {train_f1:.4f}')
    print(f'CV val   F1:     {val_f1:.4f}  (gap: {gap:.4f} → {overfit})')

    metrics_df, _ = model_kpi(name, y_test, y_pred, y_proba, show_plots=True)
    print(metrics_df.to_string(index=False))

    all_metrics.append(metrics_df)

print('\nAll 5 models trained.')

## 9. Weighted Soft-Voting Ensemble

Each model outputs a probability between 0 and 1. We combine them using **weighted soft voting** — each model's probability is weighted by its test-set F1 score, so a stronger model has more influence on the final prediction.

In [ ]:
# Collect individual F1 scores to use as ensemble weights
individual_f1 = {row['Model']: row['F1'] for df_row in all_metrics
                  for _, row in df_row.iterrows()}

print('Individual model F1 scores (used as ensemble weights):')
for name, f1 in individual_f1.items():
    print(f'  {name}: {f1:.4f}')

# Build weighted soft-voting ensemble
estimator_list = [(name.lower().replace(' ', '_'), clf)
                   for name, clf in trained_models.items()]
weights = [individual_f1[name] for name in trained_models]

ensemble = VotingClassifier(
    estimators=estimator_list,
    voting='soft',
    weights=weights
)

print('\nFitting weighted soft-voting ensemble on training data...')
ensemble.fit(X_train, y_train)
print('Ensemble trained.')

# Evaluate ensemble
y_pred_ens  = ensemble.predict(X_test)
y_proba_ens = ensemble.predict_proba(X_test)[:, 1]

metrics_ens, _ = model_kpi('Ensemble (Weighted Soft Vote)',
                            y_test, y_pred_ens, y_proba_ens, show_plots=True)
print(metrics_ens.to_string(index=False))

all_metrics.append(metrics_ens)

## 10. 6-Way Model Comparison

In [ ]:
# Build comparison DataFrame
comparison_df = pd.concat(all_metrics, ignore_index=True)
comparison_df = comparison_df.sort_values('F1', ascending=False).reset_index(drop=True)

# Highlight the winner row
winner_name = comparison_df.iloc[0]['Model']

def highlight_winner(row):
    if row['Model'] == winner_name:
        return ['background-color: #e6f4ea; font-weight: bold'] * len(row)
    return [''] * len(row)

print('=== Model Comparison — All Metrics (sorted by F1) ===')
display(comparison_df.style
        .apply(highlight_winner, axis=1)
        .format({'Accuracy': '{:.4f}', 'Precision': '{:.4f}',
                 'Recall': '{:.4f}', 'F1': '{:.4f}',
                 'ROC-AUC': '{:.4f}', 'FPR': '{:.4f}', 'FNR': '{:.4f}'})
       )

print(f'\n✓ Winner: {winner_name}')

In [ ]:
# Grouped bar chart — visual comparison of all metrics across all models
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
plot_df = comparison_df.set_index('Model')[metric_cols]

fig, ax = plt.subplots(figsize=(14, 6))
plot_df.T.plot(kind='bar', ax=ax, width=0.75,
               colormap='tab10', edgecolor='white')

ax.set_title('Model Comparison — All Metrics', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_ylim(0.75, 1.02)
ax.set_xticklabels(metric_cols, rotation=0, fontsize=11)
ax.legend(title='Model', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('graphs/07_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: graphs/07_model_comparison.png')

## 11. Save the Best Model

In [ ]:
# Select winner model object
if winner_name == 'Ensemble (Weighted Soft Vote)':
    winner_model = ensemble
else:
    winner_model = trained_models[winner_name]

# Save model and label encoder
joblib.dump(winner_model, MODEL_PATH)
joblib.dump(label_encoder, ENCODER_PATH)

print(f'Winner: {winner_name}')
print(f'Model saved  → {MODEL_PATH}')
print(f'Encoder saved → {ENCODER_PATH}')

# Quick verification
loaded_model   = joblib.load(MODEL_PATH)
loaded_encoder = joblib.load(ENCODER_PATH)
print(f'\nVerification: model type = {type(loaded_model).__name__}')

## 12. Inference — Predict a New Message

The `predict_message()` function takes a raw SMS string and returns the classification + confidence probability.

In [ ]:
def predict_message(message_text, model=None, encoder=None):
    """
    Predict whether a raw SMS message is ham or smish.

    Parameters:
        message_text  — raw SMS string
        model         — trained classifier (loads smishing_detector.pkl if None)
        encoder       — LabelEncoder (loads label_encoder.pkl if None)

    Returns:
        (label, confidence) — e.g. ('smish', 0.97)
    """
    if model is None:
        model = joblib.load(MODEL_PATH)
    if encoder is None:
        encoder = joblib.load(ENCODER_PATH)

    # Build a single-row DataFrame to reuse engineer_features
    row = pd.DataFrame([{
        'message': message_text,
        'url':     'no',
        'email':   'no',
        'phone':   'no'
    }])

    # Detect flags automatically from the message text
    feats = extract_features_from_text(message_text)
    row['url']   = feats['url']
    row['email'] = feats['email']
    row['phone'] = feats['phone']

    # Engineer text features
    row = engineer_features(row)

    # SBERT embedding
    processed = preprocess_text(message_text)
    embedding = sbert_model.encode([processed], convert_to_numpy=True)

    flags     = row[['url', 'email', 'phone']].values.astype(float)
    text_feats = row[text_feature_cols].values.astype(float)

    # Combined feature vector
    X_new = np.hstack([embedding, flags, text_feats])

    # Predict
    pred_int  = model.predict(X_new)[0]
    proba     = model.predict_proba(X_new)[0]
    label_str = encoder.inverse_transform([pred_int])[0]
    confidence = proba[pred_int]

    return label_str, round(float(confidence), 4)


# ── Example predictions ──────────────────────────────────────────────────────
test_messages = [
    'Congratulations! You have been selected for a FREE prize. Click http://claim-now.net immediately!',
    'Your bank account has been locked. Verify your password at http://secure-bank-login.com',
    'Hey, are you coming to dinner tonight? Let me know.',
    'Ok lar... Joking wif u oni...',
]

print('=== Inference Examples ===')
for msg in test_messages:
    label, conf = predict_message(msg)
    indicator = '🚨' if label == 'smish' else '✓'
    print(f'\n{indicator} [{label.upper()} | {conf:.1%}]')
    print(f'   "{msg[:80]}..."' if len(msg) > 80 else f'   "{msg}"')